## CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# import os
# import subprocess

# # Add ~/.local/bin to PATH so uv can be found
# os.environ['PATH'] = f"/home/shamouda/.local/bin:{os.environ.get('PATH', '')}"

# # Install uv
# subprocess.run("wget -qO- https://astral.sh/uv/install.sh | sh", shell=True, check=False)
# print("uv installed. Restart the kernel before proceeding.")

# # Create a virtual environment
# subprocess.run("uv venv .venv --seed", shell=True, check=False)

# # Install dependencies — this is fast thanks to uv's parallel resolver
# subprocess.run(".venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter", shell=True, check=False)

# # Install Jupyter Kernel
# subprocess.run(".venv/bin/python -m ipykernel install --user --name cse151b --display-name 'Python (cse151b)'", shell=True, check=False)

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# activate venv after installation. This needs to be run everytime.
import sys
!{sys.executable} -m pip install pandas
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician."
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
# from vllm.model_executor.models import ModelRegistry
# from vllm.model_executor.models.qwen2 import Qwen2ForCausalLM

# This bypasses the ValidationError by mapping the unknown 'Qwen3' 
# name to the 'Qwen2' logic that vLLM already understands.
# ModelRegistry.register_model("Qwen3ForCausalLM", Qwen2ForCausalLM)

os.environ["VLLM_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_FLASHINFER_FORCE_TENSOR_CORES"] = "0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.7,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
    #enforce_eager=True,
    attention_backend="TRITON_ATTN",
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.5,
    top_p=0.96,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-31 14:50:08 [utils.py:240] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'attention_backend': 'TRITON_ATTN', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


WARNING 05-31 14:50:08 [envs.py:1866] Unknown vLLM environment variable detected: VLLM_FLASHINFER_FORCE_TENSOR_CORES


INFO 05-31 14:50:09 [model.py:568] Resolved architecture: Qwen3ForCausalLM


INFO 05-31 14:50:09 [model.py:1697] Using max model len 16384


INFO 05-31 14:50:09 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=32768.


...

WARNING 05-31 14:50:44 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


(EngineCore pid=3419) 

WARNING 05-31 14:50:44 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


(EngineCore pid=3419) 

WARNING 05-31 14:50:44 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [7]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from transformers import AutoTokenizer

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"

# # 1. Configuration for A30
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16, # Optimized for A30
#     bnb_4bit_use_double_quant=True,
# )

# # 2. Load with SDPA (The built-in alternative to Flash Attention)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
#     attn_implementation="sdpa"  
# )

# # 3. Compile for extra speed (Optional, but recommended)
# # Note: The very first time you run a batch, it will take 1-2 minutes to 
# # compile. After that, it will be much faster.
# model = torch.compile(model)

# print("Model loaded successfully.")

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [8]:
# Build prompts for first 5 entries hello
prompts = []
for item in data[900:]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

    prompt_text += "</think>\n"
    
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 43 questions...


Rendering prompts:   0%|          | 0/43 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/43 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

### Generate with Transformers (for Datahub)

In [9]:
# # Group your data into batches (e.g., 8 or 16 at a time)
# BATCH_SIZE = 4
# all_responses = []

# for i in range(0, 4, BATCH_SIZE):
#     batch_items = data[i:i + BATCH_SIZE]
    
#     # Build batch of prompts
#     batch_prompts = []
#     for item in batch_items:
#         system, user = build_prompt(item["question"], item.get("options"))
#         text = tokenizer.apply_chat_template(
#             [{"role": "system", "content": system}, {"role": "user", "content": user}],
#             tokenize=False, add_generation_prompt=True
#         )
#         batch_prompts.append(text)

#     # Tokenize the whole batch at once
#     inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(model.device)

#     # Generate for the whole batch
#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             do_sample=True,
#             temperature=0.6,
#             pad_token_id=tokenizer.eos_token_id
#         )

#     # Decode
#     prompt_len = inputs.input_ids.shape[1]
#     for j, out in enumerate(output_ids):
#         generated_text = tokenizer.decode(out[prompt_len:], skip_special_tokens=True)
#         all_responses.append(generated_text.strip())

#     print(f"Batch {i // BATCH_SIZE + 1}/{(len(data) + BATCH_SIZE - 1) // BATCH_SIZE} done — {min(i + BATCH_SIZE, len(data))}/{len(data)} items processed")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [10]:
# def extract_letter(text: str) -> str:
#     m = re.search(r"\\boxed\{([A-Za-z])\}", text)
#     if m:
#         return m.group(1).upper()
#     matches = re.findall(r"\b([A-Z])\b", text.upper())
#     return matches[-1] if matches else ""


# def score_mcq(response: str, gold_letter: str) -> bool:
#     return extract_letter(response) == gold_letter.strip().upper()


# # Load Judger for free-form scoring
# sys.path.insert(0, ".")
# from judger import Judger
# judger = Judger(strict_extract=False)

# results = []
# for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
#     is_mcq = bool(item.get("options"))
#     gold   = item["answer"]

#     if is_mcq:
#         correct = score_mcq(response, str(gold))
#     else:
#         gold_list = gold if isinstance(gold, list) else [gold]
#         try:
#             correct = judger.auto_judge(
#                 pred=response,
#                 gold=gold_list,
#                 options=[[]] * len(gold_list),
#             )
#         except Exception:
#             correct = False

#     results.append({
#         "id":       item.get("id"),
#         "is_mcq":   is_mcq,
#         "gold":     gold,
#         "response": response,
#         "correct":  correct,
#     })

# print(f"Scoring complete. {len(results)} results.")

In [11]:
# For private.jsonl — no answers available, just collect responses
results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Collecting"):
    results.append({
        "id":       item.get("id"),
        "is_mcq":   bool(item.get("options")),
        "response": response,
    })

print(f"Done. {len(results)} responses collected.")

Collecting:   0%|          | 0/943 [00:00<?, ?it/s]

Collecting:   5%|▍         | 43/943 [00:00<00:00, 433545.85it/s]

Done. 43 responses collected.


## 8. Summary

Print accuracy broken down by question type.

In [12]:
# mcq_res  = [r for r in results if r["is_mcq"]]
# free_res = [r for r in results if not r["is_mcq"]]

# def acc(subset):
#     return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

# print("=" * 50)
# print("EVALUATION RESULTS")
# print("=" * 50)
# print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
# print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
# print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
# print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [13]:
SAVE_EVAL = False   # Set to False when running on the private test set

out_path = Path("results/private_results.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", newline="") as f:
    if SAVE_EVAL:
        fieldnames = ["id", "is_mcq", "gold", "response", "correct"]
    else:
        fieldnames = ["id", "is_mcq", "response"]
    
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        writer.writerow(record)

print(f"Saved {len(results)} records to {out_path}")

Saved 43 records to results/set_last_private_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [16]:
# Code used to convert a combined .jsonl file to the proper submission standard as we had to separate training into chunks

# import pandas as pd

# # Load the CSV
# df = pd.read_csv('submission3.csv')
# df.head()  # preview it
# df['id'] = range(943)  # 0 to 942
# df.to_csv('submission3.csv', index=False)

,id,response
0,0,"4, 16"
1,1,A
2,2,"-1.46, 1.645, A"
3,3,B
4,4,"-\dfrac{7\sqrt{149}}{149}, \dfrac{10\sqrt{149}..."
